# Notebook 01 — Data Understanding and Preprocessing

**Project:** Credit Portfolio Segmentation and Risk Pattern Analysis
**Covers:** Stage 1 (Data collection and understanding) and Stage 2 (Data preprocessing) of the project document.

**What this notebook does**
1. Loads the Lending Club dataset (1.35 million loans) and checks its structure, data types, missing values and duplicates.
2. Builds the **data dictionary** → `reports/data_dictionary.csv`.
3. Cleans the data: drops free-text columns, converts dates, fixes hidden missing values and a placeholder code, and checks outliers.
4. Saves the cleaned data → `data/processed/loans_clean.pkl`, and a record of every cleaning step → `reports/cleaning_log.csv`.

**How to run:** pick the `.venv` kernel (top right → *Select Kernel* → *Python Environments* → `.venv`), then click **Run All**. It takes about a minute.

> **Golden rule of this project:** the `Default` column is never used to build the clusters. In this notebook we only *look* at it.

## 0. Setup

The notebook lives in the `notebooks/` folder, but our reusable code lives in `src/`. The first lines add the project folder to Python's search path so that `from src... import ...` works. All cleaning logic is in **`src/data_processing.py`** — the notebook calls those functions and explains what they do. Keeping logic in `src/` is what the project guidelines mean by *modular functions*: the same code can be reused by later notebooks and the dashboard.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make `src` importable (this notebook runs from the notebooks/ folder).
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLEAN_DATA_FILE, CLEANING_LOG_FILE, DATA_DICTIONARY_FILE, FIGURES_DIR,
    ID_COLUMN, NUMERIC_COLUMNS, TARGET_COLUMN,
)
from src.data_processing import (
    clean_loans, describe_columns, iqr_outlier_count, load_raw_data,
)
from src.plot_style import (
    COMPACT_FORMATTER, DOLLAR_FORMATTER, PRIMARY_COLOR, TEXT_SECONDARY,
    apply_style, compact_dollars,
)

apply_style()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} "
      f"| numpy {np.__version__}")

---
# Stage 1 — Data Collection and Understanding

**Concept.** Before cleaning or modelling, we need to know exactly what is in the data: how many rows, which columns, what type each column is, what is missing, and which values look strange.

**Why it matters.** Every later decision — which columns to clean, transform, encode or exclude — is based on these facts. In the viva you must be able to justify each choice with evidence from this stage.

## 1.1 Load the data

`pd.read_csv` reads the CSV file into a **DataFrame** (a table in memory). Inside `load_raw_data()` we pass `low_memory=False`, which makes pandas read the whole file before deciding each column's type. Without it, pandas reads in chunks and warns that the `desc` column has "mixed types".

In [ ]:
raw = load_raw_data()

raw_memory_gb = raw.memory_usage(deep=True).sum() / 1e9
print(f"Rows: {raw.shape[0]:,} | Columns: {raw.shape[1]}")
print(f"Memory used: {raw_memory_gb:.2f} GB")

**How to read the output.** You should see **1,347,681 rows and 15 columns** — exactly the size of the published *Lending Club loan dataset for granting models*. Each row is one loan that has finished: it was either fully paid or it defaulted.

## 1.2 First look at the rows and the column types

We hide the two free-text columns (`title`, `desc`) in printouts: they are written by borrowers and can contain personal stories, and the project guidelines ask us to protect sensitive information.

In [ ]:
raw.drop(columns=["title", "desc"]).head()

In [ ]:
raw.dtypes

**How to read the output.**
- `int64` / `float64` are numbers (whole numbers / decimals).
- `str` is text. `issue_d` is stored as text like `Dec-2015`, so Python cannot sort it or group it by year yet — Stage 2 converts it into a real date.
- `emp_length`, `purpose`, `home_ownership_n`, `addr_state` and `zip_code` are text **categories** that will need encoding before clustering (Stage 4).

## 1.3 The data dictionary

A **data dictionary** lists every column with its type, missing values, number of distinct values, an example, its meaning and how we plan to use it. It is part of the project documentation, so we also save it to `reports/data_dictionary.csv`.

In [ ]:
data_dictionary = describe_columns(raw)

DATA_DICTIONARY_FILE.parent.mkdir(parents=True, exist_ok=True)
data_dictionary.to_csv(DATA_DICTIONARY_FILE)
data_dictionary

**How to read the output.** Look at three columns of this table:
- **`unique_values`** — `id` has 1,347,681 distinct values (one per loan), so it identifies loans but says nothing about borrowers → never a clustering feature. `experience_c` has only 2 values (more on it in 1.6).
- **`missing_pct`** — only three columns have empty cells (next step).
- **`planned_role`** — how each column will be used. Note that `Default` is marked *post-clustering analysis only*.

## 1.4 Missing values

In [ ]:
missing = data_dictionary.loc[
    data_dictionary["missing"] > 0, ["missing", "missing_pct"]
]
missing.sort_values("missing", ascending=False)

**How to read the output.**
- `desc` is **91%** empty → useless for analysis; free text is also out of scope → dropped in Stage 2.
- `title` is 1.2% empty → free text → dropped as well.
- `zip_code` has **one** empty cell → labelled `Unknown` (zip is not a model input).

⚠️ **Hidden missing values.** A value can be "missing" without being empty. Two columns use a code instead of a blank: `emp_length` uses the text **`NI`** (*no information*), and `dti_n` uses the number **999**. We find them in 1.6 and 1.7.

## 1.5 Duplicates

In [ ]:
n_duplicate_rows = raw.duplicated().sum()
n_duplicate_ids = raw[ID_COLUMN].duplicated().sum()
print(f"Duplicate rows: {n_duplicate_rows}")
print(f"Duplicate loan ids: {n_duplicate_ids}")

**How to read the output.** Both are **0**: every loan appears exactly once, so nothing needs to be removed. (We still keep the duplicate check in the cleaning pipeline, so it would catch duplicates in any future version of the file.)

## 1.6 Categorical columns

In [ ]:
for column in ["emp_length", "home_ownership_n", "purpose", "experience_c"]:
    counts = raw[column].value_counts()
    table = pd.DataFrame({
        "loans": counts,
        "percent": (counts / len(raw) * 100).round(2),
    })
    print(f"\n{column} — {raw[column].nunique()} distinct values")
    print(table.to_string())

**How to read the output.**
- **`emp_length`** has 12 values. Eleven are years (`< 1 year` … `10+ years`); **`NI` (5.8% of loans) means "no information"** — a hidden missing value. In Stage 2 we convert the text into a number of years (0–10) and keep a 0/1 flag for the `NI` borrowers, because "employment not reported" may itself describe a borrower type (we test this in the EDA).
- **`home_ownership_n`**: MORTGAGE 49%, RENT 40%, OWN 11%, OTHER only 517 loans (0.04%).
- **`purpose`**: 14 categories, but debt consolidation (58%) and credit card (22%) cover 80% of loans. Stage 4 will group the small categories.
- **`experience_c`**: **1,347,656 loans have the value 1 and only 25 have 0.** A column that is the same for 99.998% of borrowers cannot separate anyone into segments, so it is **excluded from clustering** (a *near-zero-variance* feature).

## 1.7 Numeric columns

`describe()` gives the count, mean, standard deviation, minimum, chosen percentiles and maximum. We add the **skewness**: 0 means a symmetric distribution; large positive values mean a long tail to the right (a few very large values).

In [ ]:
numeric_summary = raw[NUMERIC_COLUMNS].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T
numeric_summary["skew"] = raw[NUMERIC_COLUMNS].skew()
numeric_summary.round(2)

In [ ]:
print("DTI values equal to 999:", (raw["dti_n"] == 999).sum())
print("DTI values between 100 and 999:",
      raw["dti_n"].between(100, 999, inclusive="neither").sum())

**How to read the output.**
- **`revenue` (annual income):** median \$65,000 and 99% of borrowers earn under \$260,000 — but the maximum is about **\$11 million** and the skewness is about **45**. A handful of very high incomes stretch the scale. Without a fix, K-Means distances would be dominated by these few borrowers.
- **`dti_n` (debt-to-income %):** median about 17.6, 99% below about 38 — but the maximum is **999**. Exactly **38 loans have DTI = 999**: a placeholder code meaning "unknown", not a real ratio (→ replaced with "missing" in Stage 2). Another 495 loans lie between 100 and 999: unusual but possible, so we keep them as anomaly candidates.
- **`loan_amnt`:** \$500 to \$40,000 — the platform's limits; mild skew (about 0.8).
- **`fico_n`:** 612 to 847.5, and the 1st percentile is already 662 — Lending Club approved few borrowers below about 660, so the scores sit in a narrow range.

## 1.8 The `Default` column — look, don't touch

In [ ]:
default_counts = raw[TARGET_COLUMN].value_counts().sort_index()
default_counts.index = ["0 = fully paid", "1 = defaulted"]
print(default_counts.to_string())
print(f"\nOverall default rate: {raw[TARGET_COLUMN].mean() * 100:.2f}%")

**How to read the output.** About **1 in 5 loans (19.98%) defaulted**. This is the portfolio's baseline: in Stage 8 we compare each segment's default rate with it (for example, "segment 3 defaults at 1.5× the portfolio average").

`Default` is **never** an input to the clustering. If it were, the algorithm would group borrowers *by their outcome*, and the default-rate comparison would prove nothing — that is called **target leakage**.

## Stage 1 — summary

| Finding | Evidence | What we do (Stage 2 or later) |
|---|---|---|
| 1,347,681 loans × 15 columns, no duplicates | 1.1, 1.5 | Nothing to remove |
| `title`, `desc` are free text; `desc` is 91% empty | 1.4 | Drop both (out of scope) |
| `issue_d` is text (`Dec-2015`) | 1.2 | Convert to a real date + year + month |
| `emp_length` = `NI` for 5.8% of loans | 1.6 | Years 0–10 + a 0/1 "unknown" flag |
| `dti_n` = 999 for 38 loans | 1.7 | Replace 999 with missing (NaN) |
| `experience_c` is 1 for 99.998% of loans | 1.6 | Exclude from clustering |
| Income extremely skewed (skew ≈ 45) | 1.7 | Log-transform in Stage 4 |
| 19.98% default rate | 1.8 | Baseline for Stage 8 |

---
# Stage 2 — Data Preprocessing

**Concept.** Preprocessing turns raw data into a dataset that is correct, consistent and ready for analysis — without changing what the data says.

**Why it matters.** Clustering measures distances between borrowers. A placeholder like 999, a date stored as text, or a hidden missing value would silently distort those distances.

## 2.1 Run the cleaning pipeline

Each step is a small function in `src/data_processing.py`. `clean_loans()` runs them in order and records a **cleaning log**: what was done, to how many rows, and why. One function call reproduces all of Stage 2 — useful if new loan data arrives.

In [ ]:
clean, cleaning_log = clean_loans(raw)
cleaning_log

**How to read the output.** One row per step. `rows_affected` shows how many loans each step changed: all rows for the column-level steps, **0** duplicates, **78,188** `NI` employment values, **38** DTI placeholders and **1** missing zip code. This table is saved to `reports/cleaning_log.csv` at the end — it documents the preprocessing, as the reproducibility guideline requires.

## 2.2 Check each step worked

**Dates.** `issue_d` is now a real date with year and month columns.

In [ ]:
print("Loans issued from", clean["issue_date"].min().date(),
      "to", clean["issue_date"].max().date())
clean["issue_year"].value_counts().sort_index()

**How to read the output.** Loans run from June 2007 to December 2018, with most issued in 2014–2016. There are far fewer loans after 2016. The likely reason is how the dataset was built: it keeps only loans that had *finished* (fully paid or defaulted), so many loans issued in 2017–2018 were still running and were left out. Remember this **limitation** when comparing years in the EDA — recent years are under-represented.

**Employment length.** Each text label should map to one number of years, and only `NI` should be flagged.

In [ ]:
clean.groupby("emp_length", observed=True).agg(
    loans=("emp_length_years", "size"),
    years=("emp_length_years", "first"),
    unknown_flag=("emp_length_unknown", "max"),
)

**How to read the output.** `< 1 year` → 0, `1 year` → 1, … `10+ years` → 10, and **`NI` → NaN with `unknown_flag` = 1**. Employment length is an *ordinal* variable (its categories have a natural order), so converting it to a number keeps that order — something one-hot encoding would lose.

**DTI placeholder and remaining missing values.**

In [ ]:
print("DTI values equal to 999 now:", (clean["dti_n"] == 999).sum())
remaining_missing = clean.isna().sum()
remaining_missing[remaining_missing > 0]

**How to read the output.** No 999 values are left. Exactly two gaps remain, both **on purpose**:
- `dti_n`: 38 unknown ratios (the former 999s)
- `emp_length_years`: 78,188 unknown employment lengths (the former `NI`s)

We fill them in **Stage 4** with the median, inside the feature pipeline. Doing it there means the same fill value is saved with the model and reused by the dashboard. The 0/1 flag keeps the fact that employment was unknown.

## 2.3 Outliers — find them, but keep them

**Concept.** An **outlier** is a value far from most others. A common test is **Tukey's IQR rule**: the IQR (interquartile range) is the width of the middle 50% of the data (Q3 − Q1), and a value is an outlier if it lies more than 1.5 × IQR below Q1 or above Q3.

**Our decision: keep every loan.** These are real borrowers, not errors (the true errors — the 999 codes — are already fixed). Deleting them would remove exactly the unusual profiles that **Stage 9 (anomaly detection)** must investigate, and would bias the portfolio. Instead, Stage 4 reduces their *influence* on the clustering with transformations and capping.

The table counts IQR outliers on the original scale and on a log scale (`log1p(x)` = log(1 + x)).

In [ ]:
outlier_table = pd.DataFrame(index=NUMERIC_COLUMNS)
outlier_table["outliers_original"] = [
    iqr_outlier_count(clean[col].dropna()) for col in NUMERIC_COLUMNS
]
outlier_table["outliers_after_log"] = [
    iqr_outlier_count(np.log1p(clean[col].dropna())) for col in NUMERIC_COLUMNS
]
outlier_table["pct_original"] = (
    outlier_table["outliers_original"] / len(clean) * 100
).round(2)
outlier_table["pct_after_log"] = (
    outlier_table["outliers_after_log"] / len(clean) * 100
).round(2)
outlier_table

**How to read the output — this is evidence for Stage 4.**
- **Income:** the log scale cuts the outliers from about **4.7% to 1.7%** → income will be **log-transformed**.
- **DTI and loan amount:** the log scale makes things *worse* (more outliers), because these columns are not skewed in the way income is → they stay on the original scale, and their extreme values will be **capped** instead.
- **FICO:** no change. Its "outliers" are simply very good scores (around 790 and above) — genuine borrowers, not errors.

The lesson: a transformation is chosen *per column, based on evidence* — not applied to everything by habit. The chart below shows why income needs it.

In [ ]:
income = clean["revenue"]
fig, (ax_original, ax_log) = plt.subplots(1, 2, figsize=(12, 4.2))

ax_original.hist(income, bins=60, color=PRIMARY_COLOR, rwidth=0.9)
ax_original.set_title("Annual income — original scale")
ax_original.set_xlabel("Annual income (USD)")
ax_original.set_ylabel("Number of loans")
ax_original.xaxis.set_major_formatter(DOLLAR_FORMATTER)
ax_original.yaxis.set_major_formatter(COMPACT_FORMATTER)
ax_original.text(
    0.97, 0.95,
    f"Largest income: {compact_dollars(round(income.max(), -5))}\n"
    f"99% earn under {compact_dollars(income.quantile(0.99))}",
    transform=ax_original.transAxes, ha="right", va="top",
    color=TEXT_SECONDARY, fontsize=9,
)

log_bins = np.logspace(np.log10(income.min()), np.log10(income.max()), 60)
ax_log.hist(income, bins=log_bins, color=PRIMARY_COLOR, rwidth=0.9)
ax_log.set_xscale("log")
ax_log.set_title("Annual income — log scale")
ax_log.set_xlabel("Annual income (USD, log scale)")
ax_log.set_ylabel("Number of loans")
ax_log.xaxis.set_major_formatter(DOLLAR_FORMATTER)
ax_log.yaxis.set_major_formatter(COMPACT_FORMATTER)

fig.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "01_income_original_vs_log.png")
plt.show()

**How to read the chart.**
- **Left (original scale):** almost every loan is squeezed into the first few bars, while the axis stretches out to \$11 million for a handful of borrowers. This is what a skewness of 45 looks like.
- **Right (log scale):** the same data, spread evenly, forms a smooth hump. On this scale the gap between \$30K and \$60K counts the same as between \$100K and \$200K — differences become *relative*, which is how income really matters for credit.

That is why Stage 4 uses `log1p(revenue)` as the clustering feature. (The chart is saved to `reports/figures/` for the reports.)

## 2.4 Save the cleaned data

We save with **pickle** (`.pkl`), the format from the course's Pickling session. Unlike CSV, a pickle keeps every column's type exactly — dates stay dates and categories stay categories — so the next notebook loads it in about a second. Only open pickle files that you created yourself: loading one can run code.

In [ ]:
CLEAN_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
clean.to_pickle(CLEAN_DATA_FILE)
cleaning_log.to_csv(CLEANING_LOG_FILE, index=False)

clean_memory_gb = clean.memory_usage(deep=True).sum() / 1e9
print(f"Saved {len(clean):,} rows × {clean.shape[1]} columns to:")
print(f"  {CLEAN_DATA_FILE}")
print(f"Memory: raw {raw_memory_gb:.2f} GB → clean {clean_memory_gb:.2f} GB")
print("Cleaning log saved to:")
print(f"  {CLEANING_LOG_FILE}")

**How to read the output.** All 1,347,681 loans are kept, now in 17 columns, and the table takes far less memory, mainly because repeated labels are stored as categories. `data/processed/` is listed in `.gitignore`, so this file is not uploaded to GitHub — anyone can recreate it by running this notebook.

## 2.5 Automatic checks

An `assert` stops the notebook with an error if a condition is false. These lines prove that cleaning did what we claimed — a habit worth showing in the viva.

In [ ]:
assert len(clean) == len(raw), "Cleaning must not drop any loans"
assert clean[ID_COLUMN].is_unique, "Every loan id must appear once"
assert not {"title", "desc"} & set(clean.columns), "Text columns must be gone"
assert not (clean["dti_n"] == 999).any(), "No DTI placeholder may remain"
assert clean["emp_length_unknown"].sum() == (raw["emp_length"] == "NI").sum()
assert clean[TARGET_COLUMN].equals(raw[TARGET_COLUMN]), "Default unchanged"
print("All checks passed ✔")

## Stage 2 — summary

| Step | Loans affected | Why |
|---|---|---|
| Dropped `title`, `desc` | all (2 columns) | Free text, out of scope; `desc` 91% empty |
| Removed duplicates | 0 | None found |
| `issue_d` → `issue_date`, `issue_year`, `issue_month` | all | Text dates can't be sorted or grouped |
| `emp_length` → years + `emp_length_unknown` flag | 78,188 `NI` | Hidden missing value |
| `dti_n` 999 → NaN | 38 | Placeholder code, not a ratio |
| `zip_code` NaN → `Unknown` | 1 | Complete the column |
| Labels → `category` type | all | Memory and speed |
| Outliers | 0 removed | Real borrowers; handled in Stage 4, studied in Stage 9 |

**Next:** Notebook 02 — Exploratory Data Analysis (Stage 3).

---
## Viva practice — Stages 1 and 2

Try to answer each question out loud before reading the answer.

**1. How many records and features does your dataset have?**
1,347,681 loans and 15 columns: an id, the issue date, four numeric features (income, DTI, loan amount, FICO), a binary flag, five categorical columns, two free-text columns and the `Default` outcome.

**2. What is the default rate?**
19.98% — about one loan in five.

**3. How did you handle missing values?**
`desc` (91% empty) and `title` were dropped as out-of-scope free text. One missing zip code became `Unknown`. Two *hidden* missing values were found — `NI` in employment length and the 999 code in DTI — and converted to NaN; they are filled with the median in Stage 4, and a flag keeps the "employment unknown" information.

**4. Why didn't you delete the outliers?**
They are real borrowers, not errors. Deleting them would bias the portfolio and remove the very profiles that the anomaly-detection stage must study. Their influence on clustering is reduced with a log transform (income) and capping (DTI, loan amount).

**5. Why is `experience_c` excluded?**
It is 1 for 99.998% of loans (only 25 zeros). A near-constant column cannot separate borrowers into segments.

**6. Why convert employment length to numbers instead of one-hot encoding it?**
It is ordinal — "2 years" is between "1 year" and "3 years". A number keeps that order and uses one column instead of eleven.

**7. Why is `id` kept in the cleaned data if it is not a feature?**
For traceability: anomalies and segment members can be traced back to the original loans. It is excluded from the features in Stage 4.

**8. Why is `Default` never used in clustering?**
Clustering must find borrower groups from their characteristics only. If `Default` were an input, the groups would be built around the outcome, and comparing default rates across them would be circular (target leakage).